# Phase 4: Sales Analytics

This notebook analyses supported sales dimensions and time trends. Amazon and international sales remain separate because currency, order grain, and field coverage differ. Reported gross sales are not presented as net sales or profit.

## 1. Analytical objective

Determine how reported sales performance changes over time and which supported dimensions explain differences in reported amount, units, and order activity. The analysis is descriptive and does not infer causation.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    display = print
import duckdb
ROOT = Path.cwd().resolve()
if not (ROOT / 'data').exists(): ROOT = ROOT.parent
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from src.visualizations import apply_business_style
from src.status_scope import add_status_scope
apply_business_style()
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda value: f'{value:,.2f}')


## 2. Dataset loading and approved definitions

In [ ]:
amazon = pd.read_csv(ROOT / 'data' / 'cleaned' / 'amazon_sale_report_cleaned.csv')
international = pd.read_csv(ROOT / 'data' / 'cleaned' / 'international_sale_report_cleaned.csv')
amazon['date'] = pd.to_datetime(amazon['date'], format='%Y-%m-%d', errors='coerce')
international['date'] = pd.to_datetime(international['date'], format='%m/%d/%Y', errors='coerce')
amazon['month_period'] = amazon['date'].dt.to_period('M')
international['month_period'] = international['date'].dt.to_period('M')
amazon = add_status_scope(amazon)
delivered_proxy = amazon[amazon['is_delivered_status_proxy']].copy()
assert amazon['date'].notna().all() and international['date'].notna().all()
assert pd.api.types.is_numeric_dtype(amazon['amount']) and pd.api.types.is_numeric_dtype(amazon['qty'])
assert pd.api.types.is_numeric_dtype(international['gross_amt']) and pd.api.types.is_numeric_dtype(international['pcs'])
print('Amazon reported source:', amazon.shape, '| delivered status proxy:', delivered_proxy.shape, '| International:', international.shape)


**Gross sales definition:** Amazon `SUM(amount)` and international `SUM(gross_amt)` are reported gross-value measures kept in separate scopes.

**Net sales definition:** not calculated because discounts, refunds, and return values are unavailable.

**Order definition:** Amazon `COUNT(DISTINCT order_id)` only. International sales have no order identifier and currency is not supplied.

**Status treatment:** reported-source and delivered-status-proxy scopes are shown separately. The delivered scope is a status proxy, not a confirmed completed-sales definition. Cancelled and returned rows are not silently netted.

## 3. SQL headline reconciliation

In [ ]:
sql_con = duckdb.connect(str(ROOT / 'data' / 'processed' / 'ecommerce.duckdb'), read_only=True)
sql_monthly = sql_con.execute('SELECT sales_month, reported_amount, amount_lines, line_count, amount_coverage_pct, reported_units, distinct_orders FROM amazon_monthly_sales ORDER BY sales_month').fetchdf()
sql_international = sql_con.execute('SELECT sales_month, reported_gross_amount, reported_pieces, line_count FROM international_monthly_sales ORDER BY sales_month').fetchdf()
sql_con.close()
sql_monthly['sales_month'] = pd.to_datetime(sql_monthly['sales_month'])
sql_international['sales_month'] = pd.to_datetime(sql_international['sales_month'])
display(sql_monthly)
print('SQL Amazon amount:', sql_monthly['reported_amount'].sum())
print('SQL Amazon orders:', sql_monthly['distinct_orders'].sum())
print('SQL International gross:', sql_international['reported_gross_amount'].sum())
print(f'Amazon reported amount coverage: {amazon.amount.notna().mean():.1%}; missing values are retained and not imputed.')


## 4. Grain and coverage

In [ ]:
coverage = pd.DataFrame({'source':['Amazon reported source','Amazon delivered-status proxy','International'], 'grain':['sales line','sales line','sales line'], 'rows':[len(amazon),len(delivered_proxy),len(international)], 'distinct_order_ids':[amazon['order_id'].nunique(), delivered_proxy['order_id'].nunique(), np.nan], 'date_min':[amazon.date.min().date(),delivered_proxy.date.min().date(),international.date.min().date()], 'date_max':[amazon.date.max().date(),delivered_proxy.date.max().date(),international.date.max().date()], 'amount_coverage':[amazon.amount.notna().mean(),delivered_proxy.amount.notna().mean(),international.gross_amt.notna().mean()]})
display(coverage)
amount_coverage_by_status = amazon.groupby('status', dropna=False).agg(rows=('status','size'), amount_populated=('amount','count'), amount_coverage=('amount', lambda values: values.notna().mean()))
amount_coverage_by_month = amazon.groupby('month_period').agg(rows=('date','size'), amount_populated=('amount','count'), amount_coverage=('amount', lambda values: values.notna().mean()))
amount_coverage_by_category = amazon.groupby('category', dropna=False).agg(rows=('category','size'), amount_populated=('amount','count'), amount_coverage=('amount', lambda values: values.notna().mean()) )
display(amount_coverage_by_status, amount_coverage_by_month, amount_coverage_by_category)
print('International has no order_id and is not used for order/AOV calculations.')


## 5. Headline sales, orders, units, ASP, and AOV-style measure

In [ ]:
amazon_gross = amazon['amount'].sum()
delivered_gross = delivered_proxy['amount'].sum()
international_gross = international['gross_amt'].sum()
amazon_orders = amazon['order_id'].nunique()
delivered_orders = delivered_proxy['order_id'].nunique()
amazon_units = amazon['qty'].sum()
delivered_units = delivered_proxy['qty'].sum()
international_units = international['pcs'].sum()
amazon_asp = amazon_gross / amazon_units if amazon_units != 0 else np.nan
delivered_asp = delivered_gross / delivered_units if delivered_units != 0 else np.nan
international_asp = international_gross / international_units if international_units != 0 else np.nan
reported_value_per_distinct_order = amazon_gross / amazon_orders if amazon_orders != 0 else np.nan
delivered_value_per_distinct_order_proxy = delivered_gross / delivered_orders if delivered_orders != 0 else np.nan
headline = pd.DataFrame({'metric':['Amazon reported gross sales','Amazon delivered-status-proxy amount','International reported gross sales','Amazon distinct orders','Amazon delivered-status-proxy orders','Amazon reported units','Amazon delivered-status-proxy units','International reported pieces','Amazon reported amount per unit','Amazon delivered-status-proxy amount per unit','International reported amount per piece','Amazon reported amount per distinct order','Amazon delivered-status-proxy amount per distinct order'], 'value':[amazon_gross,delivered_gross,international_gross,amazon_orders,delivered_orders,amazon_units,delivered_units,international_units,amazon_asp,delivered_asp,international_asp,reported_value_per_distinct_order,delivered_value_per_distinct_order_proxy]})
display(headline)
display(coverage[['source', 'amount_coverage']])
print('Net sales is not calculated: no discount, refund, or return-value fields.')


## 6. Daily, weekly, and monthly trends

**Observation:** Amazon sales are concentrated in the April–June 2022 boundary window; international sales cover a longer, non-overlapping period.

**Evidence:** The tables below aggregate `amount`/`gross_amt` at daily, Monday-start weekly, and monthly grains.

**Interpretation:** The trends identify timing and mix questions, not causes.

**Business implication:** Planning and reporting should use source-specific periods and avoid combining currencies.

**Limitation:** March and June are partial boundary months, are incomplete extract periods, and must not be treated as full-month comparisons.

In [ ]:
amazon_daily = amazon.groupby('date').agg(reported_amount=('amount','sum'), amount_lines=('amount','count'), line_count=('amount','size'), amount_coverage=('amount',lambda values: values.notna().mean()), distinct_orders=('order_id','nunique'), units=('qty','sum'))
amazon_weekly = amazon.groupby(amazon['date'].dt.to_period('W-SUN')).agg(reported_amount=('amount','sum'), amount_lines=('amount','count'), line_count=('amount','size'), amount_coverage=('amount',lambda values: values.notna().mean()), distinct_orders=('order_id','nunique'), units=('qty','sum'))
amazon_monthly = amazon.groupby('month_period').agg(reported_amount=('amount','sum'), amount_lines=('amount','count'), line_count=('amount','size'), amount_coverage=('amount',lambda values: values.notna().mean()), distinct_orders=('order_id','nunique'), units=('qty','sum'))
delivered_monthly = delivered_proxy.groupby('month_period').agg(delivered_proxy_amount=('amount','sum'), amount_lines=('amount','count'), line_count=('amount','size'), amount_coverage=('amount',lambda values: values.notna().mean()), delivered_proxy_orders=('order_id','nunique'), delivered_proxy_units=('qty','sum'))
international_monthly = international.groupby('month_period').agg(reported_gross_amount=('gross_amt','sum'), gross_amount_coverage=('gross_amt',lambda values: values.notna().mean()), pieces=('pcs','sum'), line_count=('gross_amt','size'))
display(amazon_daily.head(), amazon_weekly, amazon_monthly, delivered_monthly, international_monthly)
fig, axes = plt.subplots(1, 3, figsize=(18,4))
amazon_daily['reported_amount'].plot(ax=axes[0], color='#2f6690')
axes[0].set_title('Amazon daily reported amount', loc='left', fontweight='bold'); axes[0].set_xlabel('Date'); axes[0].set_ylabel('Reported amount')
amazon_weekly['reported_amount'].plot(ax=axes[1], marker='o', color='#3a7d44')
axes[1].set_title('Amazon weekly reported amount', loc='left', fontweight='bold'); axes[1].set_xlabel('Week starting Monday'); axes[1].set_ylabel('Reported amount')
amazon_monthly['reported_amount'].plot(ax=axes[2], marker='o', color='#d1495b')
axes[2].set_title('Amazon monthly reported amount', loc='left', fontweight='bold'); axes[2].set_xlabel('Month'); axes[2].set_ylabel('Reported amount')
plt.tight_layout(); plt.show()


## 7. Month-over-month growth and partial-month controls

In [ ]:
month_flags = amazon_monthly.copy()
month_flags['is_partial_extract_month'] = False
month_flags.loc[month_flags.index.min(), 'is_partial_extract_month'] = True
month_flags.loc[month_flags.index.max(), 'is_partial_extract_month'] = True
month_flags['previous_amount'] = month_flags['reported_amount'].shift(1)
month_flags['raw_mom_growth_pct'] = month_flags['reported_amount'].pct_change() * 100
month_flags['previous_is_partial'] = month_flags['is_partial_extract_month'].shift(1).fillna(False).astype(bool)
month_flags['comparable_mom_growth_pct'] = month_flags['raw_mom_growth_pct'].where(~month_flags['is_partial_extract_month'] & ~month_flags['previous_is_partial'])
display(month_flags)
assert pd.isna(month_flags.iloc[0]['raw_mom_growth_pct'])
assert month_flags.iloc[0]['is_partial_extract_month'] and month_flags.iloc[-1]['is_partial_extract_month']
print('Only complete-month comparisons are populated in comparable_mom_growth_pct.')


## 8. Category contribution

**Observation:** Categories differ in reported amount contribution within the delivered-status-proxy scope.

**Evidence:** Contribution shares are calculated from rows with the exact status `Shipped - Delivered to Buyer`; reported-source totals remain available separately.

**Interpretation:** The ranking identifies mix within a status proxy, without implying causation or confirmed completion.

**Business implication:** Use the status-scoped result for a controlled product review and reconcile it to the source-scope result.

**Limitation:** This is reported gross amount under a status proxy, not net sales or profit.

In [ ]:
category_contribution = delivered_proxy.groupby('category', dropna=False).agg(reported_amount=('amount','sum'), amount_coverage=('amount',lambda values: values.notna().mean()), units=('qty','sum'), distinct_orders=('order_id','nunique'), line_count=('order_id','size')).sort_values('reported_amount', ascending=False)
category_contribution['amount_share'] = category_contribution['reported_amount'] / category_contribution['reported_amount'].sum()
display(category_contribution.head(10))
print('Delivered-status-proxy category share total:', category_contribution['amount_share'].sum())
assert np.isclose(category_contribution['amount_share'].sum(), 1.0)


## 9. SKU contribution and concentration

**Observation:** A small set of SKUs can be ranked by reported amount, but SKU attribution is limited by missing/unmatched identifiers.

**Evidence:** Top-five SKU contribution is calculated only over Amazon rows with a non-null SKU.

**Interpretation:** This is an assortment-concentration signal, not a profitability ranking.

**Business implication:** Top SKUs merit availability/status review; low-value SKUs should not be removed without understanding their role.

**Limitation:** Reported amount includes status-mixed lines and no validated cost.

In [ ]:
sku_contribution = delivered_proxy.dropna(subset=['sku']).groupby('sku').agg(reported_amount=('amount','sum'), amount_coverage=('amount',lambda values: values.notna().mean()), units=('qty','sum'), distinct_orders=('order_id','nunique'), line_count=('order_id','size')).sort_values('reported_amount', ascending=False)
sku_contribution['amount_share_of_delivered_status_proxy_scope'] = sku_contribution['reported_amount'] / sku_contribution['reported_amount'].sum()
top5_skus = sku_contribution.head(5)
bottom5_skus = sku_contribution[sku_contribution['reported_amount'] > 0].tail(5).sort_values('reported_amount')
display(top5_skus, bottom5_skus)
print('Delivered-status-proxy SKU share total:', sku_contribution['amount_share_of_delivered_status_proxy_scope'].sum())
assert np.isclose(sku_contribution['amount_share_of_delivered_status_proxy_scope'].sum(), 1.0)


## 10. Platform/channel, B2B, and geography

**Observation:** Amazon channel labels, B2B flags, and shipping geography are available within the Amazon extract.

**Evidence:** The tables below use distinct orders for order counts and reported amount for contribution.

**Interpretation:** These are descriptive mix comparisons, not full marketplace or customer analyses.

**Business implication:** The dimensions can guide operational segmentation and data-quality follow-up.

**Limitation:** International sales have no comparable platform, B2B, or currency fields, and shipping geography is not a customer identifier.

In [ ]:
channel_performance = amazon.groupby('sales_channel', dropna=False).agg(reported_amount=('amount','sum'), amount_coverage=('amount',lambda values: values.notna().mean()), units=('qty','sum'), distinct_orders=('order_id','nunique'), line_count=('order_id','size')).sort_values('reported_amount', ascending=False)
b2b_performance = amazon.groupby('b2b', dropna=False).agg(reported_amount=('amount','sum'), amount_coverage=('amount',lambda values: values.notna().mean()), units=('qty','sum'), distinct_orders=('order_id','nunique'), line_count=('order_id','size')).sort_values('reported_amount', ascending=False)
geography_performance = amazon.groupby('ship_state', dropna=False).agg(reported_amount=('amount','sum'), amount_coverage=('amount',lambda values: values.notna().mean()), units=('qty','sum'), distinct_orders=('order_id','nunique'), line_count=('order_id','size')).sort_values('reported_amount', ascending=False)
fulfilment_performance = amazon.groupby('fulfilment', dropna=False).agg(reported_amount=('amount','sum'), amount_coverage=('amount',lambda values: values.notna().mean()), units=('qty','sum'), distinct_orders=('order_id','nunique'), line_count=('order_id','size')).sort_values('reported_amount', ascending=False)
display(channel_performance, b2b_performance, fulfilment_performance, geography_performance.head(10))
fig, axes = plt.subplots(1, 3, figsize=(18,4))
channel_performance['reported_amount'].plot.bar(ax=axes[0], color='#486581')
axes[0].set_title('Reported amount by channel', loc='left', fontweight='bold'); axes[0].set_xlabel('Channel'); axes[0].set_ylabel('Reported amount')
b2b_performance['reported_amount'].plot.bar(ax=axes[1], color='#3a7d44')
axes[1].set_title('Reported amount by B2B flag', loc='left', fontweight='bold'); axes[1].set_xlabel('B2B'); axes[1].set_ylabel('Reported amount')
geography_performance.head(10)['reported_amount'].sort_values().plot.barh(ax=axes[2], color='#d1495b')
axes[2].set_title('Top shipping states by amount', loc='left', fontweight='bold'); axes[2].set_xlabel('Reported amount'); axes[2].set_ylabel('Ship state')
plt.tight_layout(); plt.show()


## 11. Sales concentration

**Observation:** Concentration percentages can be calculated for complete category and attributed-SKU scopes.

**Evidence:** Top-five shares are computed from the same denominator used for each scope.

**Interpretation:** High concentration indicates dependency on a small set of categories/SKUs, not necessarily superior economics.

**Business implication:** Concentrated sales should be paired with availability and status checks before planning decisions.

**Limitation:** No cost or margin field exists, so concentration is amount concentration only.

In [ ]:
concentration = pd.Series({'top_5_category_amount_share': category_contribution.head(5)['amount_share'].sum(), 'top_5_delivered_status_proxy_sku_amount_share': top5_skus['amount_share_of_delivered_status_proxy_scope'].sum()})
display(concentration.to_frame('share'))
assert ((category_contribution['amount_share'] >= 0) & (category_contribution['amount_share'] <= 1)).all()
assert ((sku_contribution['amount_share_of_delivered_status_proxy_scope'] >= 0) & (sku_contribution['amount_share_of_delivered_status_proxy_scope'] <= 1)).all()
assert np.isclose(category_contribution['amount_share'].sum(), 1.0) and np.isclose(sku_contribution['amount_share_of_delivered_status_proxy_scope'].sum(), 1.0)


## 12. Seasonality and unusual periods

**Observation:** The extract has only four Amazon calendar months and two boundary partial months.

**Evidence:** March 2022 starts on March 31 and June 2022 ends on June 29; IQR screening identifies unusual daily amounts without removing them.

**Interpretation:** The period is insufficient for a reliable seasonal pattern claim.

**Business implication:** Obtain a longer, consistently bounded history before making seasonal planning decisions.

**Limitation:** Outliers may represent bulk orders, shipping lines, or data-entry issues; investigation is required.

In [ ]:
daily_q1, daily_q3 = amazon_daily['reported_amount'].quantile([0.25,0.75])
daily_iqr = daily_q3 - daily_q1
daily_upper = daily_q3 + 1.5 * daily_iqr
unusual_days = amazon_daily[amazon_daily['reported_amount'] > daily_upper].sort_values('reported_amount', ascending=False)
display(unusual_days.head(10))
print(f'Daily IQR upper bound: {daily_upper:,.2f}; flagged days: {len(unusual_days):,}.')
print('No rows were removed for being unusual.')


## 13. Headline findings and independent reproductions

In [ ]:
independent_top5_categories = delivered_proxy.groupby('category', dropna=False)['amount'].sum().sort_values(ascending=False).head(5)
independent_top5_skus = delivered_proxy.dropna(subset=['sku']).groupby('sku')['amount'].sum().sort_values(ascending=False).head(5)
assert np.allclose(independent_top5_categories.values, category_contribution.head(5)['reported_amount'].values)
assert np.allclose(independent_top5_skus.values, top5_skus['reported_amount'].values)
print('Top five delivered-status-proxy categories and SKUs independently reproduced from direct group-by calculations.')
print('Amazon reported headline amount reconciles to SQL:', np.isclose(amazon_gross, sql_monthly['reported_amount'].sum()))
print('Amazon reported distinct orders reconcile to SQL:', amazon_orders == sql_monthly['distinct_orders'].sum())
print('Amazon reported units reconcile to SQL:', np.isclose(amazon_units, sql_monthly['reported_units'].sum()))


## 14. Questions for later phases

- Which status precedence rule should define a completed Amazon order for executive reporting?
- Are high-contribution categories/SKUs supported by stable stock and fulfilment performance?
- Why do many sales SKUs fail to match product snapshots?
- Can a longer, consistently bounded history support actual seasonality analysis?
- Can discount, refund, return-value, and cost data support net sales and profitability later?

## 15. Limitations

- Net sales is not calculated because discount, refund, and return values are unavailable.
- AOV is labelled a reported-value-per-distinct-order measure because `amount` is line-grain and its order-level treatment is not confirmed.
- International sales have no order ID, platform field, B2B flag, or currency.
- March and June 2022 are partial extract months and are excluded from like-for-like MoM percentages.
- Shipping geography is descriptive geography, not customer analysis.
- No profitability, margin, customer, return-rate, or inventory-turnover analysis is performed.

## 16. Findings and exclusions

The accompanying report records major findings with observation, evidence, interpretation, business implication, and limitation. Unsupported KPIs are explicitly excluded.

In [ ]:
print('Sales findings are documented in reports/sales_findings.md.')
print('Excluded: net sales, profit, margin, customer metrics, true return rate, inventory turnover, and causal claims.')
